## Pandas analysis

This exercise consists in analyzing a dataset containg timing information from a series of Time-to-Digital-Converters (TDC) implemented in a couple of FPGAs. Each measurement (i.e. each row of the input file) consists of a flag that specifies the type of message ('HEAD', which in this case is always 1), two addresses of the TDC providing the signal ('FPGA' and 'TDC_CHANNEL'), and the timing information ('ORBIT_CNT', 'BX_COUNTER', and 'TDC_MEAS'). Each TDC count corresponds to 25/30 ns, whereas a unit of BX_COUNTER corresponds to 25 ns, and the ORBIT_CNT is increased every 'x' BX_COUNTER. This allows to store the time in a similar way to hours, minutes and seconds.

In [1]:
# If you didn't download it yet, please get the relevant file now!
!wget https://www.dropbox.com/s/xvjzaxzz3ysphme/data_000637.txt -P ~/data/

--2026-11-19 09:08:40--  https://www.dropbox.com/s/xvjzaxzz3ysphme/data_000637.txt
Resolving www.dropbox.com (www.dropbox.com)... 162.125.69.18, 2620:100:6025:18::a27d:4512
Connecting to www.dropbox.com (www.dropbox.com)|162.125.69.18|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://www.dropbox.com/scl/fi/pkkpoxlm7beasryexpdf8/data_000637.txt?rlkey=rkm2em1v57hewglzelmin21c9 [following]
--2026-11-19 09:08:41--  https://www.dropbox.com/scl/fi/pkkpoxlm7beasryexpdf8/data_000637.txt?rlkey=rkm2em1v57hewglzelmin21c9
Reusing existing connection to www.dropbox.com:443.
HTTP request sent, awaiting response... 302 Found
Location: https://ucba3cf795b76bdb88a4d92e72d7.dl.dropboxusercontent.com/cd/0/inline/DMOKV_5j2BdZp7XAGgx0GWPhkwCfW8jBHlqBIgIwvRdUQ6JbEK3l22e_kzu_NAYPfHEjjqjKdi8FvbDLsx3L8sO8biQa3dOlC-17Yt9IfhhsvKcnhEZRphrcgtGSCzbK2hQ/file# [following]
--2026-11-19 09:08:41--  https://ucba3cf795b76bdb88a4d92e72d7.dl.dropboxusercontent.com/cd/0/inline/DMOKV_5j2B

1\. Create a Pandas DataFrame reading N rows of the 'data_000637.txt' dataset. Choose N to be smaller than or equal to the maximum number of rows and larger that 10k.

2\. Find out the number of BX in a ORBIT (the value 'x').

3\. Find out how much the data taking lasted. You can either make an estimate based on the fraction of the measurements (rows) you read, or perform this check precisely by reading out the whole dataset.

4\. Create a new column with the absolute time in ns (as a combination of the other three columns with timing information).

5\. Replace the values (all 1) of the HEAD column randomly with 0 or 1.

6\. Create a new DataFrame that contains only the rows with HEAD=1.

7\. Make two occupancy plots (one for each FPGA), i.e. plot the number of counts per TDC channel

8\. Use the groupby method to find out the noisy channels, i.e. the TDC channels with most counts (say the top 3)

9\. Count the number of unique orbits. Count the number of unique orbits with at least one measurement from TDC_CHANNEL=139

1\. Create a Pandas DataFrame reading N rows of the 'data_000637.txt' dataset. Choose N to be smaller than or equal to the maximum number of rows and larger that 10k.

In [9]:
import pandas as pd
import numpy as np

filename = '/home/contece/data/data_000637.txt'
with open(filename, 'r') as file:
    num_rows = sum(1 for line in file)

print(num_rows)

1310721


In [30]:
N = 100000 #10k<N<maxN

data=pd.read_csv(filename, nrows = N)
idx = data.index[data['ORBIT_CNT'].diff()==1]
selected_idx = idx.union(idx - 1).union(idx + 1)

data_new = data.loc[selected_idx]
max_bx = data['BX_COUNTER'].max()

print(max_bx)
print(data_new)

3563
       HEAD  FPGA  TDC_CHANNEL   ORBIT_CNT  BX_COUNTER  TDC_MEAS
42        1     0           57  3869200167        3187        17
43        1     1            1  3869200168           7        18
44        1     0           64  3869200168           7        16
127       1     0           62  3869200168        3538         4
128       1     0           74  3869200169           1        18
129       1     1           16  3869200169         131        11
254       1     0           49  3869200169        2766         1
255       1     0           44  3869200170         846        12
256       1     0           18  3869200170         848         6
352       1     1            3  3869200170        3377        11
353       1     0          124  3869200171         125        16
354       1     1            2  3869200171         262        27
461       1     0           64  3869200171        3538         7
462       1     0           42  3869200172          33         2
463       1     0   

3\. Find out how much the data taking lasted. You can either make an estimate based on the fraction of the measurements (rows) you read, or perform this check precisely by reading out the whole dataset.

In [29]:
import math

orbit_cnt_end = data['ORBIT_CNT'].iloc[-1]
orbit_cnt_start = data.loc[0, 'ORBIT_CNT']
orbit_time = 25*max_bx
frac_time = (orbit_cnt_end-orbit_cnt_start)*orbit_time
total_time = math.ceil(frac_time*num_rows/N)
print('final orbit count', orbit_cnt_end, 'initial orbit count', orbit_cnt_start, 'total time (ns)', total_time)


final orbit count 3869201161 initial orbit count 3869200167 total time (ns) 1160519583


4\. Create a new column with the absolute time in ns (as a combination of the other three columns with timing information).

In [32]:
t = 25

data['TIME(ns)'] = data['ORBIT_CNT']*orbit_time + data['BX_COUNTER']*t + data['TDC_MEAS']

print(data['TIME(ns)'])



0        344649004934901
1        344649004934902
2        344649004939378
3        344649004939494
4        344649004944550
5        344649004944579
6        344649004944839
7        344649004944925
8        344649004944896
9        344649004945232
10       344649004945154
11       344649004945194
12       344649004945343
13       344649004945323
14       344649004945253
15       344649004945450
16       344649004945214
17       344649004945285
18       344649004945404
19       344649004945451
20       344649004945264
21       344649004945260
22       344649004945298
23       344649004945515
24       344649004945419
25       344649004945464
26       344649004945514
27       344649004952571
28       344649004952551
29       344649004952620
              ...       
99970    344649093459378
99971    344649093459381
99972    344649093459650
99973    344649093459551
99974    344649093459575
99975    344649093459614
99976    344649093459551
99977    344649093459611
99978    344649093469680
